In [189]:
# Import Python packages
import pandas as pd
import numpy as np
import biom
from biom import Table
from biom.util import biom_open
from biom import load_table
import h5py

In [190]:
# Load the metadata
metadata_path = '../../../../Metadata/metadata_final_22102024.tsv'
metadata = pd.read_csv(metadata_path, sep='\t')
metadata

,SampleID,c_zone,visual_assessment_in_vivo_number_of_non_inflammatory_lesions_face,zone,sample_type,planned_study_day_of_visit,visual_assessment_in_vivo_number_of_inflammatory_lesions_face,day,subject_randomization_number,visual_assessment_in_vivo_number_of_non_inflammatory_lesions_cheek_right,...,cohort,subject_randomization_id,class,subject_ID,subject_ID_CC,zone_CC,group,severity_level,severity_group,subject_ID_x_group
0,LAMI.RD308.D16.C1,C1,not applicable,Lesional,skin,Day 16,not applicable,16,308,not applicable,...,acne,RD308,acne,PP_308,PP_308C1,Lesional_C1,Acne_L,moderate,moderate Acne_L,PP_308_Acne_L
1,LAMI.RD310.D21.C1,C1,72,Lesional,skin,Day 21,36,21,310,17,...,acne,RD310,acne,PP_310,PP_310C1,Lesional_C1,Acne_L,moderate,moderate Acne_L,PP_310_Acne_L
2,LAMI.RD305.D21.C3,C3,69,Non-lesional,skin,Day 21,26,21,305,25,...,acne,RD305,healthy,PP_305,PP_305C3,Non-lesional_C3,Acne_NL,absent,absent Acne_NL,PP_305_Acne_NL
3,LAMI.RD306.D18.C2,C2,not applicable,Lesional,skin,Day 18,not applicable,18,306,not applicable,...,acne,RD306,acne,PP_306,PP_306C2,Lesional_C2,Acne_L,low,low Acne_L,PP_306_Acne_L
4,LAMI.RD306.D7.C2,C2,90,Lesional,skin,Day 7,13,7,306,23,...,acne,RD306,acne,PP_306,PP_306C2,Lesional_C2,Acne_L,moderate,moderate Acne_L,PP_306_Acne_L
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261,LAMI.RD317.D21.C1,C1,77,Lesional,skin,Day 21,19,21,317,20,...,acne,RD317,acne,PP_317,PP_317C1,Lesional_C1,Acne_L,low,low Acne_L,PP_317_Acne_L
262,LAMI.RD001.D0.C1,C1,not applicable,Non-lesional,skin,Day 0,not applicable,0,1,not applicable,...,control,RD001,healthy,PP_1,PP_1C1,Non-lesional_C1,Healthy,absent,absent Healthy,PP_1_Healthy
263,LAMI.RD014.D14.C2,C2,not applicable,Non-lesional,skin,Day 14,not applicable,14,14,not applicable,...,control,RD014,healthy,PP_14,PP_14C2,Non-lesional_C2,Healthy,absent,absent Healthy,PP_14_Healthy
264,LAMI.RD314.D0.C1,C1,55,Lesional,skin,Day 0,31,0,314,16,...,acne,RD314,acne,PP_314,PP_314C1,Lesional_C1,Acne_L,low,low Acne_L,PP_314_Acne_L


In [191]:
def load_biom_table(biom_path):
    # Load BIOM table and convert to a DataFrame
    table = biom.load_table(biom_path)
    df = pd.DataFrame(table.matrix_data.toarray(),
                      index=table.ids(axis='observation'),
                      columns=table.ids(axis='sample'))

    
    return df

In [192]:
def df_to_biom(df, out_biom_path):
    table = Table(
        df.values,
        observation_ids=df.index.astype(str).tolist(),
        sample_ids=df.columns.astype(str).tolist()
    )

    with h5py.File(out_biom_path, "w") as h5f:
        table.to_hdf5(
            h5f,
            generated_by="column intersection filter"
        )

In [193]:
# Load tables
path_16S_V1V3 = "../from_Qiita/179426_feature-table_16S_V1V3_rare-11054_sampleIDfixed.biom"
path_16S_V4 = "../from_Qiita/174951_feature-table_16S_V4_rare-3769_sampleIDfixed.biom"

df_16S_V1V3 = load_biom_table(path_16S_V1V3)
df_16S_V4 = load_biom_table(path_16S_V4)

In [194]:
# Remove the cosmetic samples
df_16S_V1V3 = df_16S_V1V3.loc[
    :, ~df_16S_V1V3.columns.str.contains('COS', na=False)
]

# Remove any samples (cols) not in the metadata
df_16S_V1V3 = df_16S_V1V3.loc[
    :, df_16S_V1V3.columns.isin(metadata['SampleID'])
]

# View V1-V3 table
df_16S_V1V3


,LAMI.RD001.D0.C1,LAMI.RD001.D0.C2,LAMI.RD001.D14.C1,LAMI.RD001.D28.C1,LAMI.RD002.D14.C1,LAMI.RD001.D28.C2,LAMI.RD007.D0.C1,LAMI.RD002.D14.C2,LAMI.RD011.D28.C1,LAMI.RD007.D14.C1,...,LAMI.RD318.D28.C2,LAMI.RD319.D21.C1,LAMI.RD319.D28.C3,LAMI.RD319.D21.C2,LAMI.RD319.D7.C3,LAMI.RD318.D28.C3,LAMI.RD319.D14.C1,LAMI.RD319.D14.C2,LAMI.RD319.D9.C1,LAMI.RD319.D9.C2
ATTGAACGCTGGCGGCATGCTTTACACATGCAAGTCGAACGGCAGCGAAGAGAAGCTTGCTTCTCTGTCGGCGAGTGGCGAACGGGTGAGTATAGCATCGGAACGTGCCAAGTAGTGGGGGATAACCAAACGAAAGTTTGGCTAATACCG,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGGCCCTGCTTGCAGGGTACTCGAGTGGCGAACGGGTGAGTAACACGTGGGTGATCTGCCCTGCACTTCGGGATAAGCTTGGGAAACTGGGTCTAATACCGGATAG,18.0,76.0,27.0,82.0,6.0,136.0,0.0,0.0,0.0,0.0,...,2.0,0.0,6.0,2.0,0.0,1.0,7.0,1.0,0.0,7.0
GACGAACGCTGGCGGCGTGCCTAATACATGCAAGTAGAACGCTGAAGGAGGAGCTTGCTTCTCCGGATGAGTTGCGAACGGGTGAGTAACGCGTAGGTAACCTGCCTGGTAGCGGGGGATAACTATTGGAAACGATAGCTAATACCGCAT,93.0,138.0,49.0,0.0,2.0,55.0,7.0,0.0,2.0,5.0,...,0.0,1.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0
GACGAACGCTGGCGGCGTGCCTAATACATGCAAGTCGAGCGAAGTTTTTCTGGTGCTTGCACTAGAAAAACTTAGCGGCGAACGGGTGAGTAACACGTAAAGAACCTGCCTCATAGACTGGGACAACTATTGGAAACGATAGCTAATACC,66.0,24.0,41.0,46.0,0.0,36.0,2.0,0.0,0.0,7.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGGCTCCAGCTTGCTGGAGTACTCGAGTGGCGAACGGGTGAGTAACACGTGGGTAATCTGCCCTGCACTCTGGGATAAGCCTGGGAAACTGGGTCTAATACTGGAT,241.0,391.0,729.0,601.0,135.0,1168.0,17.0,90.0,151.0,67.0,...,8.0,42.0,88.0,58.0,17.0,8.0,25.0,69.0,103.0,82.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GACGAACGCTGGCGGCGTGCCTAATACATGCAAGTAGAACGCTGAAGGAGGAGCTTGCTCTTCTGGATGAGTTGCGAACGGGTGAGTAACGCGTAGGTAACCTGCCTGGTAGCGGGGGATAACTATTGGAAACGATAGCTAATACCGCAT,254.0,325.0,41.0,150.0,53.0,61.0,18.0,64.0,5.0,7.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGCTGAAGCTTGGTGCTTGCACTGGGTGGATGAGTGGCGAACGGGTGAGTAATACGTGAGTAACCTGCCCTTGACTCTGGGATAAGCCTGGGAAACTGGGTCTAATACTGG,0.0,30.0,0.0,21.0,1.0,18.0,0.0,29.0,0.0,0.0,...,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ATTGAACGCTGGCGGCATGCTTTACACATGCAAGTCGAACGGCAGCGAGGAGAAGCTTGCTTCTCTGTCGGCGAGTGGCGAACGGGTGAGTATAGCATCGGAACGTGCCAAGTAGTGGGGGATAACCAAACGAAAGTTTGGCTAATACCG,0.0,0.0,0.0,3.0,0.0,0.0,4.0,0.0,41.0,0.0,...,2619.0,2895.0,4847.0,3136.0,7981.0,1604.0,4154.0,4016.0,5902.0,4075.0
ATTGAACGCTGGCGGCAGGCTTAACACATGCAAGTCGAACGGTAGCAGGAGAAAGCTTGCTTTCTTGCTGACGAGTGGCGGACGGGTGAGTAATGCTTGGGAATCTGGCTTATGGAGGGGGATAACTACGGGAAACTGTAGCTAATACCG,44.0,60.0,24.0,30.0,0.0,10.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [195]:
# Remove the cosmetic samples
df_16S_V4 = df_16S_V4.loc[
    :, ~df_16S_V4.columns.str.contains('COS', na=False)
]

# Remove any samples (cols) not in the metadata
df_16S_V4 = df_16S_V4.loc[
    :, df_16S_V4.columns.isin(metadata['SampleID'])
]

# View V4 table 
df_16S_V4


,LAMI.RD001.D0.C1,LAMI.RD001.D14.C1,LAMI.RD004.D0.C2,LAMI.RD001.D0.C2,LAMI.RD004.D28.C1,LAMI.RD011.D0.C2,LAMI.RD001.D14.C2,LAMI.RD004.D28.C2,LAMI.RD017.D14.C2,LAMI.RD011.D14.C1,...,LAMI.RD319.D16.C1,LAMI.RD318.D16.C1,LAMI.RD319.D25.C2,LAMI.RD318.D18.C1,LAMI.RD318.D4.C2,LAMI.RD319.D16.C2,LAMI.RD319.D28.C1,LAMI.RD318.D9.C2,LAMI.RD319.D28.C2,LAMI.RD319.D2.C1
TACGTAGGGTGCGAGCGTTATCCGGAATTATTGGGCGTAAAGAGCTCGTAGGCGGTTTGTCGCGTCTGCCGTGAAAGTCCGGGGCTTAACTCCGGATCTGCGGTGGGTACGGGCAGACTAGAGTGCAGTAGGGGAGACTGGAATTCCTGG,0.0,0.0,0.0,0.0,1.0,0.0,0.0,3.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
TACGGAGGGAGCTAGCGTTGTTCGGAATTACTGGGCGTAAAGCGCGCGTAGGCGGTCTTTTAAGTCAGAGGTGAAAGCCCGGGGCTCAACCCCGGAATAGCCTTTGAAACTGGAAGACTTGAATCTTGGAGAGGTCAGTGGAATTCCGAG,0.0,1.0,2.0,1.0,1.0,0.0,0.0,9.0,0.0,0.0,...,0.0,0.0,0.0,3.0,1.0,0.0,0.0,0.0,0.0,1.0
TACGTAGGGTACTAGCGTTGTCCGGAATTATTGGGCGTAAAGAGCTCGTAGGTGGTTTGTCGCGTCTGCTGTGGAAACGTGCCGCTTAACGGTGCGCGTGCAGTGGGTACGGGCGGACTAGAGTGCAGTAGGGGAGTCTGGAATTCCTGG,11.0,123.0,0.0,4.0,5.0,0.0,50.0,0.0,0.0,9.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
TACGTAGGTGGCAAGCGTTGTCCGGATTTATTGGGCGTAAAGCGCGCGCAGGTGGTTTCTTAAGTCTGATGTGAAATCTCGCGGCTCAACCGCGAGCGGTCATTGGAAACTGGGGAACTTGAGTGCAGAAGAGGATAGTGGAATTCCAAG,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
TACGAAGGGTGCAAGCGTTAATCGGAATTACTGGGCGTAAAGCGCGCGTAGGTGGTTCGTTAAGTTGGATGTGAAAGCCCCGGGCTCAACCTGGGAACTGCATCCAAAACTGGCGAGCTAGAGTATGGCAGAGGGTGGTGGAATTTCCTG,0.0,0.0,0.0,0.0,14.0,0.0,0.0,2.0,0.0,16.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TACGTAGGGTGCAAGCGTTGTCCGGAATTACTGGGCGTAAAGAGCTCGTAGGTGGTTTGTCACGTCGTCTGTGAAATTCCACAGCTTAACTGTGGGCGTGCAGGCGATACGGGCTGACTTGAGTACTGTAGGGGTAACTGGAATTCCTGG,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
TACAGAGGGTGCAAGCGTTAATCGGAATTACTGGGCGTAAAGCGCGCGTAGGTGGTTTGTTAAGTTGGATGTGAAAGCCCCGGGCTCAACCTGGGAACTGCATCCAAAACTGGCAAGCTAGAGTACGGTAGAGGGTGGTGGAATTTCCTG,9.0,11.0,24.0,0.0,2.0,0.0,5.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
TACGTAGGTGGCAAGCGTTGTCCGGATTTATTGGGCGTAAAGCGAGCGCAGGCGGTCAATTAAGTCTGATGTGAAAGCCCCCGGCTCAACCGGGGAGGGTCATTGGAAACTGGTTGACTTGAGTGCAGAAGAGGAGAGTGGAATTCCATG,19.0,9.0,0.0,14.0,50.0,54.0,30.0,70.0,0.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
TACGGAGGGTGCAAGCGTTAATCGGAATTACTGGGCGTAAAGCGCACGCAGGCGGTCTGTTAAGTCAGATGTGAAATCCCCGGGCTTAACCTGGGAACTGCATTTGAAACTGGCAGGCTTGAGTCTCGTAGAGGGGGGTAGAATTCCAGG,0.0,0.0,19.0,0.0,51.0,0.0,0.0,2.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [197]:
# Get shared sample IDs across all datasets
shared_samples = (
    set(df_16S_V1V3.columns)
    & set(df_16S_V4.columns)
)

shared_samples = sorted(shared_samples)

print(f"Number of shared samples: {len(shared_samples)}")


Number of shared samples: 184


In [198]:
# Filter tables to same intersecton of shared samples
df_16S_V1V3_filt = df_16S_V1V3.loc[:, shared_samples]
df_16S_V4_filt = df_16S_V4.loc[:, shared_samples]

In [199]:
# Save the biom
df_to_biom(
    df_16S_V1V3_filt,
    "179426_feature-table_16S_V1V3_rare-11054_sampleIDfixed_16S-aligned.biom"
)

df_to_biom(
    df_16S_V4_filt,
    "174951_feature-table_16S_V4_rare-3769_sampleIDfixed_16S-aligned.biom"
)